# Visualize Velocity

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import adcircxtools as at
import adios4dolfinx as adx
import basix.ufl
import dolfinx as dx
import dolfinx.fem.petsc
import dolfinx.plot as dxp
import fenicsxtools as ft
import matplotlib.pyplot as plt
import matplotlib.tri
from mpi4py import MPI
import numpy as np
from petsc4py import PETSc
import pyvista as pv
import scipy.interpolate
import scipy.spatial
import ufl

## Read in mesh and set up data buffers

In [ ]:
# Get main filtered mesh
domain = adx.read_mesh('port_aransas.bp', MPI.COMM_WORLD)

In [ ]:
# Get structures for slightly larger mesh
# Used for triangular interpolation
interp_coordinates = np.load('interp_coordinates.npy')
interp_elements = np.load('interp_elements.npy')
interp_node_map = np.load('interp_node_map.npy')
interp_element_map = np.load('interp_element_map.npy')

In [ ]:
# Initialize triangulation for function interpolation
triangulation_interp = matplotlib.tri.Triangulation(
    interp_coordinates[:, 0],
    interp_coordinates[:, 1],
    interp_elements
)

In [ ]:
# Boolean masks for filtering out data points
interp_node_mask = interp_node_map != -1
interp_element_mask = interp_element_map != -1

## Visualize the velocity field time series

In [ ]:
W = dx.fem.functionspace(domain, ('Discontinuous Lagrange', 1, (domain.geometry.dim,)))
va = dx.fem.Function(W)
vb = dx.fem.Function(W)
va.interpolate(v0)
vb.interpolate(vel)
t = dx.fem.Constant(domain, 0.0) # Current simulation time
ta = dx.fem.Constant(domain, 0.0) # First time in interpolation interval
tb = dx.fem.Constant(domain, velocity_buffer_time) # Second time in interpolation interval
v = (t - ta) / (tb - ta) * vb + (tb - t) / (tb - ta) * va